In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import torch
from diffusers import AutoPipelineForImage2Image
from diffusers.utils import load_image

device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "runwayml/stable-diffusion-v1-5"

pipe = AutoPipelineForImage2Image.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    safety_checker=None
).to(device)

pipe.load_lora_weights("/content/drive/MyDrive/vangogh_lora_manual")
pipe.fuse_lora()


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recomme

model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion_img2img.StableDiffusionImg2ImgPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or aud

HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/content/drive/MyDrive/vangogh_lora_manual'. Use `repo_type` argument if needed.

In [ ]:
!pip -q install streamlit pyngrok diffusers transformers peft accelerate safetensors pillow


In [ ]:
%%writefile app.py
import os
import streamlit as st
import torch
from diffusers import AutoPipelineForImage2Image
from PIL import Image

MODEL_ID = "runwayml/stable-diffusion-v1-5"

def resize_and_center_crop(img, size=512):
    img = img.convert("RGB")
    w, h = img.size
    scale = size / min(w, h)
    new_w = int(w * scale)
    new_h = int(h * scale)
    img = img.resize((new_w, new_h), Image.LANCZOS)

    left = (new_w - size) // 2
    top = (new_h - size) // 2
    right = left + size
    bottom = top + size
    return img.crop((left, top, right, bottom))

@st.cache_resource
def load_pipe(lora_dir):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32

    pipe = AutoPipelineForImage2Image.from_pretrained(
        MODEL_ID,
        torch_dtype=dtype,
        safety_checker=None
    ).to(device)

    pipe.load_lora_weights(lora_dir)
    pipe.fuse_lora()
    return pipe

st.set_page_config(page_title="LoRA Art Style Transfer", layout="wide")
st.title("LoRA Art Style Transfer")
st.write("Upload an image and apply either Van Gogh or Anime style.")

style = st.selectbox("Choose style", ["Van Gogh", "Anime"])

if style == "Van Gogh":
    default_lora = "/content/vangogh_lora_manual"
    default_prompt = "in vgoghstyle, Van Gogh painting, expressive brush strokes, textured oil painting"
else:
    default_lora = "/content/anime_lora_manual"
    default_prompt = "anime style illustration, clean line art, colorful shading"

lora_dir = st.text_input("LoRA folder", value=default_lora)
content_prompt = st.text_input("Content prompt", value="a landscape")
strength = st.slider("Strength", 0.1, 0.9, 0.6, 0.05)
guidance_scale = st.slider("Guidance scale", 1.0, 12.0, 7.5, 0.5)
steps = st.slider("Inference steps", 10, 50, 25, 1)

uploaded = st.file_uploader("Upload image", type=["jpg", "jpeg", "png"])

if uploaded is not None:
    image = Image.open(uploaded)
    image = resize_and_center_crop(image, 512)

    col1, col2 = st.columns(2)
    with col1:
        st.subheader("Original")
        st.image(image, use_container_width=True)

    if st.button("Generate"):
        pipe = load_pipe(lora_dir)
        prompt = f"{content_prompt}, {default_prompt}"

        result = pipe(
            prompt=prompt,
            image=image,
            strength=strength,
            guidance_scale=guidance_scale,
            num_inference_steps=steps
        ).images[0]

        with col2:
            st.subheader(f"{style} Result")
            st.image(result, use_container_width=True)

        st.caption(f"Prompt: {prompt}")
